# Downloading the data

Download GSE181774_DSB_Repair_Map.xlsx from

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE181774

Put it in CROP/data/ALDIT

# Versions

In [16]:
import sys
import os
import re
import pandas as pd
print("Python version:", sys.version)
print("Pandas version:", pd.__version__)



Python version: 3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]
Pandas version: 2.3.0


# Preprocess

In [17]:
DATA_PATH = "../data"
EXCEL_PATH = "../data/ALDIT/GSE181774_DSB_Repair_Map.xlsx"


# Load all sheets
excel_file = pd.ExcelFile(EXCEL_PATH)
for sheet_name in excel_file.sheet_names:
    df = excel_file.parse(sheet_name)
    # Sanitize filename
    safe_name = sheet_name.replace("/", "_").replace("\\", "_")
    csv_path = os.path.join(DATA_PATH, f"{safe_name}.csv")
    df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")


Saved: ../data\K562.csv
Saved: ../data\Jurkat.csv
Saved: ../data\H1.csv
Saved: ../data\K562.DNTTOE.csv
Saved: ../data\Jurkat.DNTTKO.csv


In [18]:
PATH_K562 = "../data/K562.csv"
PATH_JURKAT = "../data/Jurkat.csv"
PATH_HAP1 = "../data/H1.csv"

def make_length_only_variant(path):
    df = pd.read_csv(path)
    columns = df.columns.tolist()

    # Group 1bp and 2bp insertions
    insertion_1 = [col for col in columns if col.startswith("I-1_")]
    insertion_2 = [col for col in columns if col.startswith("I-2_")]

    # Keep insertions I-3 to I-9+ individually

    # Detect all exact-length insertions (e.g., I-3, I-4, ...)
    insertion_exact = [
        col for col in columns
        if re.fullmatch(r"I-\d+", col) and int(col[2:]) >= 3
    ]

    # Detect the grouped 3+ insertions column (e.g., I-3+, I-4+, ..., I-9+)
    insertion_plus = [col for col in columns if re.fullmatch(r"I-\d+\+", col)]

    # Combine all
    insertion_individual = insertion_exact + insertion_plus

    print(f"Insertions grouped: {len(insertion_1)} 1bp, {len(insertion_2)} 2bp,")
    print(f"Individual insertions: {len(insertion_individual)}")

    # Deletions grouped by length (1–30), D30+ kept separate
    deletion = [col for col in columns if "D-" in col or col == "D30+"]
    deletion_groups = {}
    for col in deletion:
        if col == "D30+":
            deletion_groups.setdefault("del_30plus", []).append(col)
        else:
            try:
                length = int(col.split("D-")[-1])
                if 1 <= length <= 30:
                    group = f"del_{length}"
                    deletion_groups.setdefault(group, []).append(col)
            except ValueError:
                continue

    # Start new DataFrame with ID columns
    base_cols = ['sgRNA_name', 'gRNASeq(up20bp+target20bp+pam3bp+down42bp)']
    new_df = df[base_cols].copy()

    # Add grouped insertions
    new_df["1"] = df[insertion_1].sum(axis=1)
    new_df["2"] = df[insertion_2].sum(axis=1)

    for col in insertion_individual:
        new_col = col.replace("I-", "")  # e.g., I-3 → 3, I-3+ → 3+
        new_df[new_col] = df[col]

    # Add grouped deletions
    for group, cols in deletion_groups.items():
        if group == "del_30plus":
            new_col = "-30+"
        else:
            length = group.replace("del_", "")
            new_col = f"-{length}"
        new_df[new_col] = df[cols].sum(axis=1)


    # add a constant column for PAM position
    new_df["PAM_position"] = 40  # PAM is at position 40 in the gRNA sequence (pos 0 is the first base)

    # Save the new DataFrame
    new_df.to_csv(path.replace(".csv", "_only_length.csv"), index=False)

make_length_only_variant(PATH_K562)
make_length_only_variant(PATH_JURKAT)
make_length_only_variant(PATH_HAP1)

Insertions grouped: 4 1bp, 16 2bp,
Individual insertions: 1
Insertions grouped: 4 1bp, 16 2bp,
Individual insertions: 7
Insertions grouped: 4 1bp, 16 2bp,
Individual insertions: 1


In [19]:


PATH_JURKAT_KO = "../data/Jurkat.DNTTKO.csv"
PATH_K562_OE   = "../data/K562.DNTTOE.csv"

def load_orig_sequences(path):
    """
    Loads original dataset and returns:
      dict: sgRNA_name -> sequence
    """
    df = pd.read_csv(path)
    seq_col = "gRNASeq(up20bp+target20bp+pam3bp+down42bp)"
    return dict(zip(df["sgRNA_name"], df[seq_col]))


def make_length_only_variant_DNT(path_new, seq_dict):
    """
    Preprocess DNTTKO / DNTTOE datasets by making a length-only
    representation of insertions/deletions.
    - Ignores mismatch (M-*) columns
    - Keeps D-1..D-50 as separate lengths: -1..-50
    - Keeps I-1..I-50 as separate lengths: 1..50
    """
    df = pd.read_csv(path_new)
    columns = df.columns.tolist()

    # -------------------------------
    #  REMOVE Mismatch (M-) columns
    # -------------------------------
    mismatch_cols = [c for c in columns if c.startswith("M-")]
    df = df.drop(columns=mismatch_cols)
    columns = df.columns.tolist()

    # -------------------------------
    #   INSERTION COLUMNS (I-x)
    # -------------------------------
    ins_cols = [c for c in columns if c.startswith("I-")]

    insertion_1 = [c for c in ins_cols if c == "I-1"]
    insertion_2 = [c for c in ins_cols if c == "I-2"]

    # I-3 .. I-9
    insertion_3_9  = [c for c in ins_cols if re.fullmatch(r"I-[3-9]", c)]

    # I-10 .. I-50
    insertion_10_50 = [
        c for c in ins_cols
        if re.fullmatch(r"I-\d+", c) and 10 <= int(c[2:]) <= 50
    ]

    insertion_individual = insertion_3_9 + insertion_10_50

    # -------------------------------
    #   DELETION COLUMNS (D-x)
    #   NOW: keep D-1..D-50 individually
    # -------------------------------
    del_cols = [c for c in columns if c.startswith("D-")]

    # map length -> list of cols (usually one col each)
    deletion_groups = {}
    for col in del_cols:
        try:
            length = int(col.split("D-")[-1])  # e.g. "D-17" -> 17
        except ValueError:
            continue
        group = f"del_{length}"
        deletion_groups.setdefault(group, []).append(col)

    # -----------------------------------
    #     BUILD NEW CLEAN DATAFRAME
    # -----------------------------------
    new_df = pd.DataFrame()
    new_df["sgRNA_name"] = df["sgRNA_name"]

    # attach sequence from original dataset
    new_df["gRNASeq(up20bp+target20bp+pam3bp+down42bp)"] = \
        new_df["sgRNA_name"].map(seq_dict)

    # -------------------------------
    #  Add insertions
    # -------------------------------
    new_df["1"] = df[insertion_1].sum(axis=1) if insertion_1 else 0
    new_df["2"] = df[insertion_2].sum(axis=1) if insertion_2 else 0

    for col in insertion_individual:
        new_col = col.replace("I-", "")  # "I-3" -> "3"
        new_df[new_col] = df[col]

    # -------------------------------
    #    Add deletions: -1..-50
    # -------------------------------
    for group, cols in deletion_groups.items():
        length = group.replace("del_", "")  # "del_17" -> "17"
        new_col = f"-{length}"
        new_df[new_col] = df[cols].sum(axis=1)

    # PAM index fixed
    new_df["PAM_position"] = 40

    # Save
    out_path = path_new.replace(".csv", "_only_length.csv")
    new_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")


# ========= RUN FOR BOTH NEW DATASETS =========

seq_jurkat = load_orig_sequences(PATH_JURKAT)
seq_k562   = load_orig_sequences(PATH_K562)
seq_H1     = load_orig_sequences(PATH_HAP1)


# combined the dicts
seq_combined = {}
for (k, v) in seq_jurkat.items():
    seq_combined[k] = v
for (k, v) in seq_k562.items():
    seq_combined[k] = v

for (k, v) in seq_H1.items():
    seq_combined[k] = v

all_keys = set(seq_jurkat.keys()).union(set(seq_k562.keys())).union(set(seq_H1.keys()))
for key in all_keys:
    seqs = []
    if key in seq_jurkat:
        seqs.append(seq_jurkat[key])
    if key in seq_k562:
        seqs.append(seq_k562[key])
    if key in seq_H1:
        seqs.append(seq_H1[key])
    if len(set(seqs)) > 1:
        print(f"Warning: Different sequences for sgRNA_name {key}: {seqs}")
    


        

make_length_only_variant_DNT(PATH_JURKAT_KO, seq_combined)
make_length_only_variant_DNT(PATH_K562_OE,   seq_combined)



C:\Users\tzion\AppData\Local\Temp\ipykernel_1844\628266622.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df[new_col] = df[cols].sum(axis=1)
C:\Users\tzion\AppData\Local\Temp\ipykernel_1844\628266622.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df[new_col] = df[cols].sum(axis=1)
C:\Users\tzion\AppData\Local\Temp\ipykernel_1844\628266622.py:96: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

Saved: ../data/Jurkat.DNTTKO_only_length.csv


C:\Users\tzion\AppData\Local\Temp\ipykernel_1844\628266622.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df[new_col] = df[cols].sum(axis=1)
C:\Users\tzion\AppData\Local\Temp\ipykernel_1844\628266622.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df[new_col] = df[cols].sum(axis=1)
C:\Users\tzion\AppData\Local\Temp\ipykernel_1844\628266622.py:96: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining al

Saved: ../data/K562.DNTTOE_only_length.csv


In [21]:
# load the 2 new datasets and remove rows where sequence is NaN
path_jr_only_length=  PATH_JURKAT_KO.replace(".csv", "_only_length.csv")
path_k562_only_length= PATH_K562_OE.replace(".csv", "_only_length.csv")
df_jurkat_dnt = pd.read_csv(path_jr_only_length)
df_jurkat_dnt = df_jurkat_dnt.dropna(subset=["gRNASeq(up20bp+target20bp+pam3bp+down42bp)"])
df_k562_dnt = pd.read_csv(path_k562_only_length)
df_k562_dnt = df_k562_dnt.dropna(subset=["gRNASeq(up20bp+target20bp+pam3bp+down42bp)"])

# save cleaned datasets to the same path 
df_jurkat_dnt.to_csv(path_jr_only_length, index=False)
df_k562_dnt.to_csv(path_k562_only_length, index=False)
print(f"Cleaned and saved: {path_jr_only_length}")
print(f"Cleaned and saved: {path_k562_only_length}")

Cleaned and saved: ../data/Jurkat.DNTTKO_only_length.csv
Cleaned and saved: ../data/K562.DNTTOE_only_length.csv


In [23]:
renaming_dict = {
    "K562_only_length.csv": "ALDIT_K562.csv",
    "Jurkat_only_length.csv": "ALDIT_Jurkat.csv",
    "H1_only_length.csv": "ALDIT_HAP1.csv",
    "Jurkat.DNTTKO_only_length.csv": "ALDIT_Jurkat_DNTTKO.csv",
    "K562.DNTTOE_only_length.csv": "ALDIT_K562_DNTTOE.csv",
}

for original_name, new_name in renaming_dict.items():
    original_path = os.path.join(DATA_PATH, original_name)
    new_path = os.path.join(DATA_PATH, new_name)
    os.rename(original_path, new_path)
    print(f"Renamed: {original_path} -> {new_path}")

Renamed: ../data\K562_only_length.csv -> ../data\ALDIT_K562.csv
Renamed: ../data\Jurkat_only_length.csv -> ../data\ALDIT_Jurkat.csv
Renamed: ../data\H1_only_length.csv -> ../data\ALDIT_HAP1.csv
Renamed: ../data\Jurkat.DNTTKO_only_length.csv -> ../data\ALDIT_Jurkat_DNTTKO.csv
Renamed: ../data\K562.DNTTOE_only_length.csv -> ../data\ALDIT_K562_DNTTOE.csv


In [24]:
# open all of these csvs and remove empty columns
for filename in renaming_dict.values():
    path = os.path.join(DATA_PATH, filename)
    df = pd.read_csv(path)
    # Remove empty columns
    df_clean = df.dropna(axis=1, how='all')
    df_clean.to_csv(path, index=False)
    print(f"Cleaned empty columns: {path}")

Cleaned empty columns: ../data\ALDIT_K562.csv
Cleaned empty columns: ../data\ALDIT_Jurkat.csv
Cleaned empty columns: ../data\ALDIT_HAP1.csv
Cleaned empty columns: ../data\ALDIT_Jurkat_DNTTKO.csv
Cleaned empty columns: ../data\ALDIT_K562_DNTTOE.csv


In [25]:
# Delete intermediate files
files_to_delete = ["K562.csv", "Jurkat.csv", "H1.csv",
                   "Jurkat.DNTTKO.csv", "K562.DNTTOE.csv"]

for filename in files_to_delete:
    path = os.path.join(DATA_PATH, filename)
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted intermediate file: {path}")
    else:
        print(f"File not found, skipping deletion: {path}")

Deleted intermediate file: ../data\K562.csv
Deleted intermediate file: ../data\Jurkat.csv
Deleted intermediate file: ../data\H1.csv
Deleted intermediate file: ../data\Jurkat.DNTTKO.csv
Deleted intermediate file: ../data\K562.DNTTOE.csv


In [26]:
# Rename PAM_position to PAM position in all ALDIT files and gRNASeq(up20bp+target20bp+pam3bp+down42bp) to sequence
for filename in renaming_dict.values():
    path = os.path.join(DATA_PATH, filename)
    df = pd.read_csv(path)
    if "PAM_position" in df.columns:
        df = df.rename(columns={"PAM_position": "PAM position"})
    if "gRNASeq(up20bp+target20bp+pam3bp+down42bp)" in df.columns:
        df = df.rename(columns={"gRNASeq(up20bp+target20bp+pam3bp+down42bp)": "sequence"})
    df.to_csv(path, index=False)
    print(f"Renamed columns in: {path}")

Renamed columns in: ../data\ALDIT_K562.csv
Renamed columns in: ../data\ALDIT_Jurkat.csv
Renamed columns in: ../data\ALDIT_HAP1.csv
Renamed columns in: ../data\ALDIT_Jurkat_DNTTKO.csv
Renamed columns in: ../data\ALDIT_K562_DNTTOE.csv
